In [ ]:
!pip install hygese

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 1.2 MB/s eta 0:00:00


In [ ]:
!pip install osmnx folium networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 2.1 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import networkx as nx
import numpy as np
import hygese as hgs
import folium
import random

# 1. Setup Location and Network
location_point = (28.6129, 77.2295) # India Gate, New Delhi
radius = 3000 # 3km

# Download the drive network
G = ox.graph_from_point(location_point, dist=radius, network_type='drive')
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

# 2. Generate 26 points (1 depot + 25 deliveries)
nodes = list(G.nodes())
selected_nodes = random.sample(nodes, 26)
depot_node = selected_nodes[0]
delivery_nodes = selected_nodes[1:]
all_points = [depot_node] + delivery_nodes

# 3. Calculate Distance Matrix (Shortest Path in meters)
dist_matrix = np.zeros((26, 26))
for i in range(26):
    for j in range(26):
        if i == j:
            dist_matrix[i][j] = 0
        else:
            dist_matrix[i][j] = nx.shortest_path_length(G, all_points[i], all_points[j], weight='length')

# 4. Prepare Solver Data with requested parameters
num_vehicles = 5
vehicle_capacity = 25
data = dict()
data['distance_matrix'] = dist_matrix.tolist()
# Assign random demands so total demand < total capacity (5 * 25 = 125)
data['demands'] = [0] + [random.randint(2, 6) for _ in range(25)]
data['vehicle_capacity'] = vehicle_capacity
data['num_vehicles'] = num_vehicles
data['depot'] = 0
data['service_times'] = np.zeros(26)

# Solve
ap = hgs.AlgorithmParameters(timeLimit=1)
hgs_solver = hgs.Solver(parameters=ap, verbose=False)
result = hgs_solver.solve_cvrp(data)

# Detailed Output Display
print(f"--- CVRP Optimization Summary ---")
print(f"Total Route Distance: {result.cost:.2f} meters")
print(f"Number of Vehicles: {num_vehicles}")
print(f"Vehicle Capacity: {vehicle_capacity}")
print("\nVehicle Routing Breakdown:")
for idx, route in enumerate(result.routes):
    route_demand = sum(data['demands'][i] for i in route)
    print(f"Truck {idx + 1}: Path {' -> '.join(map(str, [0] + route + [0]))} | Load: {route_demand}/{vehicle_capacity}")

--- CVRP Optimization Summary ---
Total Route Distance: 67533.15 meters
Number of Vehicles: 5
Vehicle Capacity: 25

Vehicle Routing Breakdown:
Truck 1: Path 0 -> 16 -> 10 -> 11 -> 20 -> 15 -> 8 -> 0 | Load: 25/25
Truck 2: Path 0 -> 5 -> 7 -> 21 -> 25 -> 2 -> 19 -> 0 | Load: 25/25
Truck 3: Path 0 -> 18 -> 17 -> 6 -> 1 -> 13 -> 3 -> 0 | Load: 23/25
Truck 4: Path 0 -> 22 -> 9 -> 23 -> 24 -> 12 -> 4 -> 0 | Load: 25/25
Truck 5: Path 0 -> 14 -> 0 | Load: 5/25


In [ ]:
def visualize_routes_interactive(G, routes, all_points):
    depot_node = all_points[0]
    depot_lat, depot_lon = G.nodes[depot_node]['y'], G.nodes[depot_node]['x']

    # Initialize map
    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=14, tiles='CartoDB positron')

    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue']

    # Add Markers for all points
    for idx, node_id in enumerate(all_points):
        lat, lon = G.nodes[node_id]['y'], G.nodes[node_id]['x']
        if idx == 0:
            folium.Marker([lat, lon], popup="Depot", icon=folium.Icon(color='black', icon='home')).add_to(m)
        else:
            folium.Marker([lat, lon], popup=f"Delivery {idx}", icon=folium.Icon(color='gray')).add_to(m)

    # Add PolyLines for each route
    for vehicle_idx, route in enumerate(routes):
        color = colors[vehicle_idx % len(colors)]
        # Build the sequence of nodes: Depot -> Deliveries -> Depot
        route_nodes = [0] + route + [0]

        # Follow actual street geometry for the lines
        for i in range(len(route_nodes)-1):
            u, v = all_points[route_nodes[i]], all_points[route_nodes[i+1]]
            sp = nx.shortest_path(G, u, v, weight='length')
            sp_coords = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in sp]
            folium.PolyLine(sp_coords, color=color, weight=4, opacity=0.8, popup=f"Truck {vehicle_idx+1}").add_to(m)

    return m

# Generate the map
interactive_map = visualize_routes_interactive(G, result.routes, all_points)
interactive_map

In [ ]:
# @title
# A CVRP from https://developers.google.com/optimization/routing/cvrp
import numpy as np
import hygese as hgs

data = dict()
data['distance_matrix'] = [
    [0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354, 468, 776, 662],
    [548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674, 1016, 868, 1210],
    [776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164, 1130, 788, 1552, 754],
    [696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822, 1164, 560, 1358],
    [582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708, 1050, 674, 1244],
    [274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628, 514, 1050, 708],
    [502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856, 514, 1278, 480],
    [194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320, 662, 742, 856],
    [308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662, 320, 1084, 514],
    [194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388, 274, 810, 468],
    [536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764, 730, 388, 1152, 354],
    [502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114, 308, 650, 274, 844],
    [388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194, 536, 388, 730],
    [354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0, 342, 422, 536],
    [468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536, 342, 0, 764, 194],
    [776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274, 388, 422, 764, 0, 798],
    [662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730, 536, 194, 798, 0]
]
data['num_vehicles'] = 4
data['depot'] = 0
data['demands'] = [0, 1, 1, 2, 4, 2, 4, 8, 8, 1, 2, 1, 2, 4, 4, 8, 8]
data['vehicle_capacity'] = 15  # different from OR-Tools: homogeneous capacity
data['service_times'] = np.zeros(len(data['demands']))

# Solver initialization
ap = hgs.AlgorithmParameters(timeLimit=3.2)  # seconds
hgs_solver = hgs.Solver(parameters=ap, verbose=True)

# Solve
result = hgs_solver.solve_cvrp(data)
print(result.cost)
print(result.routes)

6208.0
[[14, 16, 10, 9], [5, 2, 6, 8], [1, 4, 3, 7], [13, 15, 11, 12]]


In [ ]:
# @title
# A CVRP from https://developers.google.com/optimization/routing/cvrp
import numpy as np
import hygese as hgs

data = dict()
data['distance_matrix'] = [
    [0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354, 468, 776, 662],
    [548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674, 1016, 868, 1210],
    [776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164, 1130, 788, 1552, 754],
    [696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822, 1164, 560, 1358],
    [582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708, 1050, 674, 1244],
    [274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628, 514, 1050, 708],
    [502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856, 514, 1278, 480],
    [194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320, 662, 742, 856],
    [308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662, 320, 1084, 514],
    [194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388, 274, 810, 468],
    [536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764, 730, 388, 1152, 354],
    [502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114, 308, 650, 274, 844],
    [388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194, 536, 388, 730],
    [354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0, 342, 422, 536],
    [468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536, 342, 0, 764, 194],
    [776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274, 388, 422, 764, 0, 798],
    [662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730, 536, 194, 798, 0]
]
data['num_vehicles'] = 4
data['depot'] = 0
data['demands'] = [0, 1, 1, 2, 4, 2, 4, 8, 8, 1, 2, 1, 2, 4, 4, 8, 8]
data['vehicle_capacity'] = 15  # different from OR-Tools: homogeneous capacity
data['service_times'] = np.zeros(len(data['demands']))

# Solver initialization
ap = hgs.AlgorithmParameters(timeLimit=3.2)  # seconds
hgs_solver = hgs.Solver(parameters=ap, verbose=True)

# Solve
result = hgs_solver.solve_cvrp(data)
print(result.cost)
print(result.routes)